In [1]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.append('../../scripts')

In [3]:
import numpy as np
import scanpy as sc
import os
DATA_ROOT = '/data/a330d' #os.environ.get("DATA_ROOT", ".")
import matplotlib.pyplot as plt
import decoupler as dc
import scipy.sparse as sp
import pandas as pd
from scgraph import scGraph


from utils import set_seed
from train_loo import preprocess_crc, preprocess_merfish, _load_model, split_indices, preprocess_spatial_features
from counterfactual_analysis import compute_rmse, compute_edistance, mixing_index, get_lfc, precision, direction_match, compute_mse_lfc, _to_dense
from counterfactual_analysis import get_perturbation_logfc, get_global_perturbation_logfc
from configs.adata_crc_config import ADATA_ARGS as ADATA_ARGS_CRC
from configs.adata_merfish_config import ADATA_ARGS as ADATA_ARGS_MERFISH
from configs.cellina_config import MODEL_ARGS as CELLINA_MODEL_ARGS, TRAIN_ARGS as CELLINA_TRAIN_ARGS, PLAN_KWARGS as CELLINA_PLAN_KWARGS
from configs.cpa_config import MODEL_ARGS as CPA_MODEL_ARGS, TRAIN_ARGS as CPA_TRAIN_ARGS, PLAN_KWARGS as CPA_PLAN_KWARGS

/data/a330d/miniforge3/envs/cellina-graph/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import cellina

cellina.__version__

'0.7.4'

In [5]:
set_seed(0)

In [6]:
DATASET_NAME = "crc"  # or "merfish"
CELLINA_BASE_MODEL_ROOT = os.path.join(DATA_ROOT, "data/ood/trained")
CELLINA_GAT_MODEL_ROOT = os.path.join(DATA_ROOT, "data/ood/trained")

In [50]:
CRC_PATHS = [
    os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_231.h5ad"),
    os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_232.h5ad"),
    os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_242.h5ad"),
]

MERFISH_PATHS = [
    os.path.join(DATA_ROOT, "datasets/MERFISH_mouse_brain/C57BL6J-2.036.h5ad"),    
    #os.path.join(DATA_ROOT, "datasets/MERFISH_mouse_brain/C57BL6J-2.039.h5ad"),
    #os.path.join(DATA_ROOT, "datasets/MERFISH_mouse_brain/C57BL6J-2.041.h5ad"),
]


PATHS = CRC_PATHS if DATASET_NAME == "crc" else MERFISH_PATHS
DATA_ARGS = ADATA_ARGS_CRC if DATASET_NAME == "crc" else ADATA_ARGS_MERFISH
COUNTS_PER_K = 1e4

In [51]:
n_top_genes = DATA_ARGS.get('n_top_genes')
labels_key = DATA_ARGS.get('labels_key')
domains_key = DATA_ARGS.get('domains_key')
batch_key = DATA_ARGS.get('batch_key')
control_domain = DATA_ARGS.get('control_domains')[0]
holdout_domains = DATA_ARGS.get('holdout_domains')
n_neighbors = DATA_ARGS.get('n_neighbors')
batch_size = 512
library_size = 'latent'
n_deg = 50
n_pert_genes = 200

In [52]:
# Create SLIDES which contain file names from PATHS - first split by "/" and take last part, then split by "." and take first part
SLIDES = [path.split("/")[-1].split(".h5ad")[0] for path in PATHS]

In [45]:
save_path = '/data/a330d/datasets/scgraph'

In [ ]:
train_models = False
plot_umaps = False
model_names = ['cellina'] #['cellina, cpa']#,
os.makedirs(f"../../figures/scgraph", exist_ok=True)

for path, slide_id in zip(PATHS, SLIDES):
    adata = sc.read(path)
    
    if DATASET_NAME == 'crc':
        adata = preprocess_crc(adata, n_top_genes=n_top_genes, labels_key=labels_key, domains_key=domains_key)
    elif DATASET_NAME == 'merfish':
        adata = preprocess_merfish(adata, n_top_genes=n_top_genes, labels_key=labels_key, domains_key=domains_key)
    else:
        raise ValueError(f"Unknown dataset_name: {DATASET_NAME}. Supported: crc, merfish")
    
    # 50 times * in print
    print(f"{'='*50} Slide: {slide_id} {'='*50}")

    # Compute spatial features after splitting to avoid data leakage
    step_size_px = 0.12028 if DATASET_NAME == 'crc' else 0.109
    adata = preprocess_spatial_features(adata, step_size_px=step_size_px, n_neighbors=n_neighbors, test_indices=None)
    
    for model_name in model_names:
        if model_name == 'cellina':
            model_class = "cellina"
        elif model_name == 'cpa':
            model_class = "cpa"
        else:
            model_class = "cellina_graph"
        
        MODEL_ROOT = CELLINA_BASE_MODEL_ROOT if model_class == 'cellina' else CELLINA_GAT_MODEL_ROOT
        save_dir = os.path.join(MODEL_ROOT, slide_id, model_name)
        
        if train_models:
            if model_class == 'cellina':
                from cellina import Cellina as CellinaModel
                model_args = CELLINA_MODEL_ARGS.copy()
                train_args = CELLINA_TRAIN_ARGS.copy()
                plan_kwargs = CELLINA_PLAN_KWARGS.copy()
                CellinaModel.setup_anndata(adata, 
                                        batch_key=batch_key, 
                                        labels_key=labels_key, 
                                        domains_key=domains_key, 
                                        spatial_obsm_key='spatial_x', 
                                        layer='counts')
                model = CellinaModel(adata, **model_args)

                if plan_kwargs is not None:
                    model.train(**train_args, plan_kwargs=plan_kwargs)
                else:
                    model.train(**train_args)
                model.save(save_dir, overwrite=True)
            if model_class == 'cpa':
                import cpa
                model_args = CPA_MODEL_ARGS.copy()
                train_args = CPA_TRAIN_ARGS.copy()
                plan_kwargs = CPA_PLAN_KWARGS.copy()

                adata.obs['dose'] = 1.0 # NOTE: dummy dose for compatibility with CPA model
                adata.obs['data_split'] = 'train'
                cpa.CPA.setup_anndata(adata,
                        perturbation_key=domains_key,
                        control_group='REF',
                        dosage_key='dose',
                        categorical_covariate_keys=[labels_key],
                        is_count_data=True,
                        max_comb_len=1,
                        )
                model = cpa.CPA(adata,
                                split_key='data_split',
                                train_split='train',
                                valid_split='valid',
                                test_split='test',
                                **model_args)
                model.train(**train_args, plan_kwargs=plan_kwargs, save_path=save_dir)
        try:
            if model_class == 'cellina':
                from cellina import Cellina as CellinaModel
                model = CellinaModel.load(save_dir, adata)
            if model_class == 'cpa':
                import cpa
                adata.obs['dose'] = 1.0 # NOTE: dummy dose for compatibility with CPA model
                adata.obs['data_split'] = 'train'
                model = cpa.CPA.load(dir_path=save_dir,
                            adata=adata,
                            use_gpu=True)
        except Exception as e:
            print(f"Failed to load model from {save_dir} with error: {e}")
            continue

        # Compute latents and store in adata.obsm
        if model_class == 'cellina':
            adata.obsm[f'X_cellina_z'] = model.get_latent_representation(latent_key='z', batch_size=batch_size)
            adata.obsm[f'X_cellina_s'] = model.get_latent_representation(latent_key='s', batch_size=batch_size)

            # Subsample 20% of data for UMAPs
            adata = adata[np.random.choice(adata.shape[0], size=int(0.2 * adata.shape[0]), replace=False)]

            sc.pp.neighbors(adata, use_rep='X_cellina_z')
            sc.tl.umap(adata)
            plt.figure(figsize=(6, 6))
            sc.pl.umap(adata, color=[domains_key, labels_key], use_raw=False, show=False)            
            plt.savefig(f"../../figures/scgraph/{slide_id}_cellina_z_umap.png", dpi=300)
            adata.obsm['X_cellina_z_umap'] = adata.obsm.pop('X_umap')

            sc.pp.neighbors(adata, use_rep='X_cellina_s')
            sc.tl.umap(adata)
            plt.figure(figsize=(6, 6))
            sc.pl.umap(adata, color=[domains_key, labels_key], use_raw=False, show=False)
            plt.savefig(f"../../figures/scgraph/{slide_id}_cellina_s_umap.png", dpi=300)
            adata.obsm['X_cellina_s_umap'] = adata.obsm.pop('X_umap')

        if model_class == 'cpa':
            latents = model.get_latent_representation(adata=adata, batch_size=batch_size)
            latents = latents["latent_after"].X
            adata.obsm['X_cpa'] = latents
            sc.pp.neighbors(adata, use_rep='X_cpa')
            sc.tl.umap(adata)
            adata.obsm['X_cpa_umap'] = adata.obsm.pop('X_umap')

    if plot_umaps:
        continue
    # Drop a list of keys from adata.obsm
    sc.pp.normalize_per_cell(adata, counts_per_cell_after=1e4)
    sc.pp.log1p(adata)

    adata_write_path = f"{save_path}/{slide_id}.h5ad"

    # Load adata_write_path if it exists, otherwise write adata to adata_write_path
    if os.path.exists(adata_write_path):
        adata_out = sc.read(adata_write_path)
        adata_out.obsm.update(adata.obsm)
        adata_out.write_h5ad(adata_write_path)
    else:
        adata.write_h5ad(adata_write_path)
        adata_out = adata

In [53]:
for slide_id in SLIDES:
    scgraph = scGraph(
        adata_path=f"{save_path}/{slide_id}.h5ad",   # Path to AnnData object
        batch_key=batch_key,                     # Column name for batch information
        label_key=labels_key,                     # Column name for cell type labels
        trim_rate=0.05,                          # Trim rate for robust mean calculation
        thres_batch=100,                       # Minimum number of cells per batch
        thres_celltype=10,                       # Minimum number of cells per cell type
        only_umap=True,                          # Only evaluate 2D embeddings (mostly umaps)
    )

    # Run the analysis, return a pandas dataframe
    results = scgraph.main()

    # Save the results
    results.to_csv(f"{save_path}/{slide_id}_embedding_evaluation_results.csv")

    print(results)

Processing batches, calcualte centroids and pairwise distances


100%|██████████| 1/1 [00:02<00:00,  2.37s/it]


                  Rank-PCA  Corr-PCA  Corr-Weighted
X_cellina_s_umap  0.576531  0.600508       0.430797
X_cellina_z_umap  0.515306  0.746446       0.258637
X_cpa_umap        0.545918  0.663651       0.264369
spatial           0.382653  0.543698       0.136566
Processing batches, calcualte centroids and pairwise distances


100%|██████████| 1/1 [00:01<00:00,  1.13s/it]


                  Rank-PCA  Corr-PCA  Corr-Weighted
X_cellina_s_umap  0.709184  0.742124       0.431159
X_cellina_z_umap  0.760204  0.836565       0.618169
X_cpa_umap        0.250000  0.540859      -0.141300
spatial           0.571429  0.565956       0.225338
Processing batches, calcualte centroids and pairwise distances


100%|██████████| 1/1 [00:04<00:00,  4.75s/it]


                  Rank-PCA  Corr-PCA  Corr-Weighted
X_cellina_s_umap  0.598214  0.675204       0.488370
X_cellina_z_umap  0.577381  0.788717       0.443495
X_cpa_umap        0.485119  0.588344       0.219513
spatial           0.622024  0.562886       0.486603


In [63]:
# Add a column for slide_id to the results dataframe when concatenating results from multiple slides
results_summary = pd.concat([pd.read_csv(f"{save_path}/{slide_id}_embedding_evaluation_results.csv", 
                                         index_col=0).assign(slide_id=slide_id) for slide_id in SLIDES], axis=0)


In [65]:
# Only keep indices where 'cellina' or 'cpa' is in the index
results_summary = results_summary[results_summary.index.str.contains('cpa|cellina')]

In [ ]:
# Create 'model' column and assign index
results_summary['model'] = results_summary.index

In [ ]:
metrics = ['Rank-PCA', 'Corr-PCA', 'Corr-Weighted']
# For each metric, compute mean and std across 'model'
agg = results_summary.groupby('model')[metrics].agg(['mean', 'std'])
table = pd.DataFrame({
    m: agg[(m, 'mean')].round(2).astype(str) + " ± " + agg[(m, 'std')].round(2).astype(str)
    for m in metrics
})
print(table)

                     Rank-PCA     Corr-PCA Corr-Weighted
model                                                   
X_cellina_s_umap  0.63 ± 0.07  0.67 ± 0.07   0.45 ± 0.03
X_cellina_z_umap  0.62 ± 0.13  0.79 ± 0.05   0.44 ± 0.18
X_cpa_umap        0.43 ± 0.16   0.6 ± 0.06   0.11 ± 0.22


In [81]:
latex_table = table.to_latex(index=True, escape=True)
print(latex_table)

\begin{tabular}{llll}
\toprule
 & Rank-PCA & Corr-PCA & Corr-Weighted \\
model &  &  &  \\
\midrule
X\_cellina\_s\_umap & 0.63 ± 0.07 & 0.67 ± 0.07 & 0.45 ± 0.03 \\
X\_cellina\_z\_umap & 0.62 ± 0.13 & 0.79 ± 0.05 & 0.44 ± 0.18 \\
X\_cpa\_umap & 0.43 ± 0.16 & 0.6 ± 0.06 & 0.11 ± 0.22 \\
\bottomrule
\end{tabular}

